In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q networkx pyvis pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 49.3 MB/s eta 0:00:00


In [ ]:
# ============================================================
# CELL 1: Install and imports
# ============================================================
# !pip install -q networkx pyvis pandas numpy

import json
import warnings
import pandas as pd
import numpy as np
import networkx as nx
from pathlib import Path
from datetime import datetime
from pyvis.network import Network

warnings.filterwarnings("ignore")

BASE_PATH   = Path("/content/drive/Othercomputers/My PC/data_u")   # CHANGE THIS
OUTPUT_PATH = Path("/content/drive/MyDrive/IntelliSense/ml_outputs")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print("✅ Imports loaded")


✅ Imports loaded


In [ ]:
# ============================================================
# CELL 2: Load all graph input data
# ============================================================
MCA_PATH   = BASE_PATH / "external intelligence" / "mca"
LEGAL_PATH = BASE_PATH / "external intelligence" / "legal_disputes"

directors_df = pd.read_csv(MCA_PATH / "mca_directors.csv")
network_df   = pd.read_csv(MCA_PATH / "director_company_network.csv")
charges_df   = pd.read_csv(MCA_PATH / "mca_charges_registered.csv")
charges_live = pd.read_csv(MCA_PATH / "mca_charges_live_only.csv")
company_df   = pd.read_csv(MCA_PATH / "mca_company_master.csv")
cases_df     = pd.read_csv(LEGAL_PATH / "company_cases.csv") if (LEGAL_PATH / "company_cases.csv").exists() else pd.DataFrame()
lit_summary  = pd.read_csv(LEGAL_PATH / "litigation_risk_summary_entity.csv") if (LEGAL_PATH / "litigation_risk_summary_entity.csv").exists() else pd.DataFrame()

print(f"📂 Data loaded:")
print(f"   Directors          : {len(directors_df):,} rows")
print(f"   Director network   : {len(network_df):,} rows")
print(f"   Charges registered : {len(charges_df):,} rows")
print(f"   Charges live       : {len(charges_live):,} rows")
print(f"   Company master     : {len(company_df):,} rows")
print(f"   Legal cases        : {len(cases_df):,} rows")
print(f"   Litigation summary : {len(lit_summary):,} rows")

# Quick column preview
for name, df in [("directors", directors_df), ("network", network_df),
                  ("charges", charges_df), ("company_master", company_df)]:
    print(f"\n   [{name}] columns: {list(df.columns)}")


📂 Data loaded:
   Directors          : 6,714 rows
   Director network   : 6,714 rows
   Charges registered : 4,476 rows
   Charges live       : 2,238 rows
   Company master     : 2,238 rows
   Legal cases        : 746 rows
   Litigation summary : 126 rows

   [directors] columns: ['director_din', 'director_name', 'company_cin', 'appointment_date', 'resignation_date', 'designation', 'din_status', 'company_id', 'case_id']

   [network] columns: ['director_din', 'company_cin', 'connection_type', 'is_borrower_company', 'other_company_risk_flags', 'company_id', 'case_id']

   [charges] columns: ['charge_id', 'company_cin', 'charge_holder_name', 'charge_amount_inr', 'charge_creation_date', 'charge_modification_date', 'charge_status', 'asset_description', 'charge_type', 'company_id', 'case_id', 'encumbrance_eligible']

   [company_master] columns: ['company_cin', 'company_name', 'company_status', 'date_of_incorporation', 'company_category', 'authorized_capital_inr', 'paid_up_capital_inr', 're

In [ ]:
# ============================================================
# CELL 3: Build the Risk Graph
# ============================================================

G = nx.MultiDiGraph()   # directed, multi-edge (same pair can have multiple relationships)

# ── NODE TYPES ────────────────────────────────────────────────────────────────
# company  → color: #3b82f6 (blue)
# director → color: #f59e0b (amber)
# bank     → color: #22c55e (green)
# case     → color: #ef4444 (red)

# ── 1. Add Company Nodes ───────────────────────────────────────────────────────
for _, row in company_df.iterrows():
    cin    = str(row.get("company_cin", ""))
    name   = str(row.get("company_name", cin))
    status = str(row.get("company_status", "active")).lower()
    is_risky = status not in ["active"]

    G.add_node(cin,
        node_type   = "company",
        label       = name[:30],
        cin         = cin,
        status      = status,
        is_risky    = is_risky,
        color       = "#ef4444" if is_risky else "#3b82f6",
        size        = 15,
    )

# ── 2. Add Director Nodes + Directorship Edges ────────────────────────────────
for _, row in directors_df.iterrows():
    din    = str(row.get("director_din", ""))
    name   = str(row.get("director_name", din))
    cin    = str(row.get("company_cin", ""))
    active = row.get("resignation_date") is None or str(row.get("resignation_date", "")).strip() in ["", "nan", "None"]
    status = str(row.get("din_status", "active")).lower()

    if not G.has_node(din):
        G.add_node(din,
            node_type = "director",
            label     = name[:25],
            din       = din,
            din_status= status,
            color     = "#ef4444" if status == "disqualified" else "#f59e0b",
            size      = 12,
        )

    if cin and G.has_node(cin):
        edge_type = "current_directorship" if active else "past_directorship"
        G.add_edge(din, cin,
            edge_type   = edge_type,
            label       = edge_type.replace("_", " "),
            color       = "#f59e0b" if active else "#666666",
            weight      = 2 if active else 1,
        )

# ── 3. Add Bank Nodes + Charge Edges ──────────────────────────────────────────
for _, row in charges_df.iterrows():
    cin          = str(row.get("company_cin", ""))
    bank_name    = str(row.get("charge_holder_name", "Unknown Bank"))
    bank_id      = f"BANK_{bank_name[:20].replace(' ', '_').upper()}"
    amount       = float(row.get("charge_amount_inr", 0) or 0)
    status       = str(row.get("charge_status", "live")).lower()
    is_live      = status == "live"

    if not G.has_node(bank_id):
        G.add_node(bank_id,
            node_type = "bank",
            label     = bank_name[:20],
            color     = "#22c55e",
            size      = 10,
        )

    if cin and G.has_node(cin):
        G.add_edge(cin, bank_id,
            edge_type    = "charge",
            charge_status= status,
            amount_inr   = amount,
            label        = f"₹{amount/1e7:.1f}Cr" if amount > 1e7 else "charge",
            color        = "#ef4444" if is_live else "#888888",
            weight       = 3 if is_live else 1,
        )

# ── 4. Add Legal Case Nodes + Edges ───────────────────────────────────────────
if not cases_df.empty:
    for _, row in cases_df.iterrows():
        cin       = str(row.get("company_cin", ""))
        case_num  = str(row.get("case_number", ""))
        case_type = str(row.get("case_type", "civil")).lower()
        status    = str(row.get("case_status", "active")).lower()
        amount    = float(row.get("amount_in_dispute_inr", 0) or 0)
        is_active = status in ["active", "pending"]
        is_severe = case_type in ["drt", "nclt", "criminal"]

        case_id = f"CASE_{case_num[:15]}"
        if not G.has_node(case_id):
            G.add_node(case_id,
                node_type  = "case",
                label      = f"{case_type.upper()}\n{case_num[:10]}",
                case_type  = case_type,
                case_status= status,
                amount_inr = amount,
                color      = "#ef4444" if (is_active and is_severe) else "#f97316",
                size       = 8,
            )

        if cin and G.has_node(cin):
            G.add_edge(cin, case_id,
                edge_type   = "litigation",
                case_type   = case_type,
                is_active   = is_active,
                label       = case_type.upper(),
                color       = "#ef4444" if is_active else "#888888",
                weight      = 3 if is_severe and is_active else 1,
            )

print(f"✅ Graph built:")
print(f"   Nodes: {G.number_of_nodes():,}  (companies, directors, banks, cases)")
print(f"   Edges: {G.number_of_edges():,}")

node_types = {}
for n, d in G.nodes(data=True):
    t = d.get("node_type", "unknown")
    node_types[t] = node_types.get(t, 0) + 1
print(f"   Node breakdown: {node_types}")


✅ Graph built:
   Nodes: 9,484  (companies, directors, banks, cases)
   Edges: 11,936
   Node breakdown: {'company': 2238, 'director': 6714, 'bank': 5, 'case': 527}


In [ ]:
# ============================================================
# CELL 4: Compute risk scores
# ============================================================

def get_company_directors(cin: str) -> list[str]:
    """Return list of director DINs for a given company CIN."""
    return [
        n for n in G.predecessors(cin)
        if G.nodes[n].get("node_type") == "director"
    ]

def get_director_companies(din: str) -> list[str]:
    """Return list of company CINs a director is connected to."""
    return [
        n for n in G.successors(din)
        if G.nodes[n].get("node_type") == "company"
    ]

def company_has_risk_flags(cin: str) -> dict:
    """Check a company node for risk signals."""
    node  = G.nodes.get(cin, {})
    flags = {
        "is_inactive"    : node.get("status", "active") not in ["active"],
        "has_drt_case"   : False,
        "has_nclt_case"  : False,
        "has_live_charge": False,
    }
    # Check outgoing edges for cases and charges
    for _, target, data in G.out_edges(cin, data=True):
        if data.get("edge_type") == "litigation":
            if data.get("case_type") == "drt" and data.get("is_active"):
                flags["has_drt_case"] = True
            if data.get("case_type") == "nclt" and data.get("is_active"):
                flags["has_nclt_case"] = True
        if data.get("edge_type") == "charge" and data.get("charge_status") == "live":
            flags["has_live_charge"] = True
    return flags

# ── Director Stress Score ──────────────────────────────────────────────────────
director_scores = []

for node_id, node_data in G.nodes(data=True):
    if node_data.get("node_type") != "director":
        continue

    other_companies = get_director_companies(node_id)
    if not other_companies:
        continue

    risky_count  = 0
    total_count  = len(other_companies)
    flag_details = []

    for cin in other_companies:
        flags = company_has_risk_flags(cin)
        n_flags = sum(flags.values())
        if n_flags > 0:
            risky_count += 1
            flag_details.append({cin: flags})

    stress_score = (risky_count / total_count) * 10   # 0-10 scale
    din_status   = node_data.get("din_status", "active")
    if din_status == "disqualified":
        stress_score = min(10.0, stress_score + 3.0)   # disqualified DIN penalty

    director_scores.append({
        "director_din"              : node_id,
        "director_name"             : node_data.get("label", ""),
        "din_status"                : din_status,
        "total_directorships"       : total_count,
        "risky_directorships"       : risky_count,
        "director_stress_score"     : round(stress_score, 4),   # 0-10
        "director_stress_pct"       : round(risky_count / total_count * 100, 1),
        "risk_flag_details"         : json.dumps(flag_details[:5]),   # top 5
        "computed_timestamp"        : datetime.utcnow().isoformat(),
    })

director_stress_df = pd.DataFrame(director_scores)
dir_out = OUTPUT_PATH / "director_stress_scores.csv"
director_stress_df.to_csv(dir_out, index=False)
print(f"💾 Saved: {dir_out}  ({len(director_stress_df)} directors scored)")


# ── Per-Company Graph Risk Rollup ──────────────────────────────────────────────
graph_risk_rows = []

for cin in [n for n, d in G.nodes(data=True) if d.get("node_type") == "company"]:
    company_flags = company_has_risk_flags(cin)
    own_directors = get_company_directors(cin)

    # Director stress: average stress of all directors of this company
    dir_stress_vals = []
    for din in own_directors:
        match = director_stress_df[director_stress_df["director_din"] == din]
        if not match.empty:
            dir_stress_vals.append(float(match.iloc[0]["director_stress_score"]))
    avg_dir_stress = np.mean(dir_stress_vals) if dir_stress_vals else 0.0

    # Live charges total
    live_charges_total = sum(
        data.get("amount_inr", 0)
        for _, _, data in G.out_edges(cin, data=True)
        if data.get("edge_type") == "charge" and data.get("charge_status") == "live"
    )

    # Network centrality (degree)
    degree = G.degree(cin)

    # Contagion: are any companies connected via shared directors in trouble?
    connected_company_flags = []
    for din in own_directors:
        for other_cin in get_director_companies(din):
            if other_cin != cin:
                connected_company_flags.append(company_has_risk_flags(other_cin))
    contagion_risk = sum(
        1 for f in connected_company_flags if any(f.values())
    ) / max(1, len(connected_company_flags))

    # Composite graph risk score (0-10, feeds Character C)
    risk_components = {
        "inactive_company"  : 3.0 if company_flags["is_inactive"] else 0.0,
        "drt_case"          : 4.0 if company_flags["has_drt_case"] else 0.0,
        "nclt_case"         : 3.0 if company_flags["has_nclt_case"] else 0.0,
        "director_stress"   : avg_dir_stress * 0.3,
        "contagion_risk"    : contagion_risk * 2.0,
    }
    graph_risk_score = min(10.0, sum(risk_components.values()))

    company_name = G.nodes[cin].get("label", cin)
    graph_risk_rows.append({
        "company_cin"               : cin,
        "company_name"              : company_name,
        "graph_risk_score"          : round(graph_risk_score, 4),   # 0-10
        "company_inactive_flag"     : company_flags["is_inactive"],
        "has_active_drt_case"       : company_flags["has_drt_case"],
        "has_active_nclt_case"      : company_flags["has_nclt_case"],
        "has_live_charges"          : company_flags["has_live_charge"],
        "live_charges_total_inr"    : round(live_charges_total, 2),
        "n_directors"               : len(own_directors),
        "avg_director_stress_score" : round(avg_dir_stress, 4),
        "contagion_risk_score"      : round(contagion_risk, 4),
        "network_degree"            : degree,
        "risk_components"           : json.dumps(risk_components),
        "computed_timestamp"        : datetime.utcnow().isoformat(),
    })

graph_risk_df = pd.DataFrame(graph_risk_rows)
risk_out = OUTPUT_PATH / "graph_risk_scores.csv"
graph_risk_df.to_csv(risk_out, index=False)
print(f"💾 Saved: {risk_out}  ({len(graph_risk_df)} companies scored)")



💾 Saved: /content/drive/MyDrive/IntelliSense/ml_outputs/director_stress_scores.csv  (6714 directors scored)
💾 Saved: /content/drive/MyDrive/IntelliSense/ml_outputs/graph_risk_scores.csv  (2238 companies scored)


In [ ]:
# ============================================================
# CELL 5: Generate interactive HTML visualization (Ugro demo)
# ============================================================

def build_focused_graph(target_cin: str, G: nx.MultiDiGraph, hops: int = 2) -> nx.MultiDiGraph:
    """Extract ego subgraph around a target company for demo visualization."""
    nodes_to_include = {target_cin}
    frontier = {target_cin}

    for _ in range(hops):
        new_frontier = set()
        for node in frontier:
            new_frontier |= set(G.predecessors(node)) | set(G.successors(node))
        nodes_to_include |= new_frontier
        frontier = new_frontier

    return G.subgraph(nodes_to_include).copy()

# Find Ugro Capital CIN
ugro_cin = None
for cin, data in G.nodes(data=True):
    if "ugro" in data.get("label", "").lower() or "ugro" in cin.lower():
        ugro_cin = cin
        break

if ugro_cin is None:
    # Fallback: use the company with highest graph risk score for demo
    if not graph_risk_df.empty:
        ugro_cin = graph_risk_df.nlargest(1, "graph_risk_score").iloc[0]["company_cin"]
        print(f"  ⚠️  Ugro not found — using highest-risk company: {ugro_cin}")
    else:
        ugro_cin = list(G.nodes())[0]
        print(f"  ⚠️  Using first company as fallback: {ugro_cin}")
else:
    print(f"  ✅ Ugro Capital CIN found: {ugro_cin}")

sub_G = build_focused_graph(ugro_cin, G, hops=2)
print(f"  Focused subgraph: {sub_G.number_of_nodes()} nodes, {sub_G.number_of_edges()} edges")

# Build Pyvis network
net = Network(
    height      = "750px",
    width       = "100%",
    bgcolor     = "#0a0a0a",
    font_color  = "#ffffff",
    directed    = True,
    notebook    = False,
)
net.set_options("""
{
  "nodes": {
    "font": {"color": "#ffffff", "size": 11},
    "borderWidth": 2
  },
  "edges": {
    "arrows": {"to": {"enabled": true, "scaleFactor": 0.5}},
    "font": {"color": "#888888", "size": 9},
    "smooth": {"type": "curvedCW", "roundness": 0.2}
  },
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -60,
      "centralGravity": 0.01,
      "springLength": 100
    },
    "solver": "forceAtlas2Based",
    "stabilization": {"iterations": 150}
  },
  "interaction": {"hover": true, "tooltipDelay": 100}
}
""")

# Add nodes
for node_id, data in sub_G.nodes(data=True):
    node_type = data.get("node_type", "unknown")
    color     = data.get("color", "#888888")
    size      = data.get("size", 10)
    label     = data.get("label", node_id[:15])

    # Highlight target company
    if node_id == ugro_cin:
        color = "#ffffff"
        size  = 30

    tooltip_parts = [f"ID: {node_id}", f"Type: {node_type}"]
    if node_type == "company":
        tooltip_parts.append(f"Status: {data.get('status', 'N/A')}")
    elif node_type == "director":
        tooltip_parts.append(f"DIN Status: {data.get('din_status', 'N/A')}")
    elif node_type == "case":
        tooltip_parts.append(f"Case Type: {data.get('case_type', 'N/A')}")
        tooltip_parts.append(f"Status: {data.get('case_status', 'N/A')}")

    net.add_node(
        node_id,
        label   = label,
        color   = color,
        size    = size,
        title   = "\n".join(tooltip_parts),
    )

# Add edges
for src, dst, data in sub_G.edges(data=True):
    edge_type = data.get("edge_type", "")
    color     = data.get("color", "#444444")
    label     = data.get("label", "")[:20]
    width     = data.get("weight", 1)
    net.add_edge(src, dst, color=color, label=label, width=width, title=edge_type)

html_out = str(OUTPUT_PATH / "graph_visualization.html")
net.write_html(html_out)
print(f"💾 Saved: {html_out}  (open in browser for interactive graph)")


  ✅ Ugro Capital CIN found: L86097MN2010PLC002079
  Focused subgraph: 983 nodes, 1786 edges
💾 Saved: /content/drive/MyDrive/IntelliSense/ml_outputs/graph_visualization.html  (open in browser for interactive graph)


In [ ]:
# ============================================================
# CELL 6: Ugro Capital graph risk spot-check
# ============================================================
print("\n" + "="*60)
print("UGRO CAPITAL GRAPH RISK SPOT-CHECK")
print("="*60)

ugro_risk = graph_risk_df[graph_risk_df["company_cin"] == ugro_cin]
if not ugro_risk.empty:
    r = ugro_risk.iloc[0]
    print(f"  Company CIN              : {r['company_cin']}")
    print(f"  Graph Risk Score         : {r['graph_risk_score']:.2f} / 10.0")
    print(f"  Company Active           : {not r['company_inactive_flag']}")
    print(f"  Active DRT Case          : {r['has_active_drt_case']} ⚠️" if r["has_active_drt_case"] else f"  Active DRT Case          : {r['has_active_drt_case']}")
    print(f"  Active NCLT Case         : {r['has_active_nclt_case']}")
    print(f"  Live Charges Total       : ₹{r['live_charges_total_inr']/1e7:.1f} Cr")
    print(f"  Number of Directors      : {r['n_directors']}")
    print(f"  Avg Director Stress      : {r['avg_director_stress_score']:.2f} / 10.0")
    print(f"  Contagion Risk Score     : {r['contagion_risk_score']:.4f}")
    print(f"\n  Risk Components:")
    for k, v in json.loads(r["risk_components"]).items():
        print(f"    {k:25s}: {v:.2f}")
else:
    print(f"  CIN {ugro_cin} not found in graph_risk_df")

print(f"\n✅ Graph Risk Engine complete.")
print(f"""
OUTPUT SUMMARY:
  director_stress_scores.csv   → Per-director stress scores (0-10)
  graph_risk_scores.csv        → Per-company graph risk scores (feeds Character C)
  graph_visualization.html     → Interactive demo visualization

FIVE Cs INTEGRATION:
  Character C ← graph_risk_score (weight ~15-20% of Character C)
                director_stress_score (signals promoter group risk)
                has_active_drt_case (CRITICAL flag)
  Collateral C ← live_charges_total_inr (encumbered collateral)
                 cross-charges detected via shared banks
  Conditions C ← contagion_risk_score (connected entity distress)
""")



UGRO CAPITAL GRAPH RISK SPOT-CHECK
  Company CIN              : L86097MN2010PLC002079
  Graph Risk Score         : 3.00 / 10.0
  Company Active           : True
  Active DRT Case          : False
  Active NCLT Case         : False
  Live Charges Total       : ₹59.4 Cr
  Number of Directors      : 3
  Avg Director Stress      : 10.00 / 10.0
  Contagion Risk Score     : 0.0000

  Risk Components:
    inactive_company         : 0.00
    drt_case                 : 0.00
    nclt_case                : 0.00
    director_stress          : 3.00
    contagion_risk           : 0.00

✅ Graph Risk Engine complete.

OUTPUT SUMMARY:
  director_stress_scores.csv   → Per-director stress scores (0-10)
  graph_risk_scores.csv        → Per-company graph risk scores (feeds Character C)
  graph_visualization.html     → Interactive demo visualization
 
FIVE Cs INTEGRATION:
  Character C ← graph_risk_score (weight ~15-20% of Character C)
                director_stress_score (signals promoter group risk)
   

In [ ]:
import pickle

# Save the graph
graph_out = OUTPUT_PATH / "graph_risk_engine_model"/ "risk_graph.pkl"
with open(graph_out, "wb") as f:
    pickle.dump(G, f)
print(f"✅ Graph saved: {graph_out}")

# Load it back later
with open(graph_out, "rb") as f:
    G = pickle.load(f)
print(f"✅ Graph loaded: {G.number_of_nodes()} nodes")


✅ Graph saved: /content/drive/MyDrive/IntelliSense/ml_outputs/graph_risk_engine_model/risk_graph.pkl
✅ Graph loaded: 9484 nodes
